In [1]:
# 用 requests 复现第一页请求
import requests
import warnings
warnings.filterwarnings("ignore")

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

api_url = "https://115.159.159.82/api/devices?page=1&page_size=50"

r = requests.get(
    api_url,
    params={
        # 按实际请求填写分页参数
        "page": 1, "page_size": 50
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50&page=1&page_size=50
content-type: application/json


In [2]:
# 读取 JSON
data = r.json()

print("type:", type(data))
print("keys:", data.keys())

type: <class 'dict'>
keys: dict_keys(['page', 'page_size', 'total', 'items'])


In [3]:
# 继续观察
print(data)

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [4]:
# 查看一条真实设备记录
print(data["items"][0])

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [5]:
# 计算总页数并采集全部设备，根据接口返回的总条数和每页条数计算：
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
# 然后完成分页：
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page, "page_size": page_size
        },
        timeout=10,
    )
    
    page_data = r.json()
    
    # 将本页设备加入 devices
    # TODO
    devices.extend(page_data["items"])
    
print("device count:", len(devices))

device count: 5000


In [7]:
# 采集 Company 静态页面，直接请求 HTML：
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

In [8]:
cards = soup.select("article.company-card")

print("本页企业数:", len(cards))


本页企业数: 20


In [9]:
# 先解析第一条
card = cards[0]

In [10]:
# 行业、区域、设备数量等信息采用 dt/dd 结构，先转换成字典：
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [11]:
# company_id 来自 data-* 属性
company_id = card.get("data-company-id")

# name 位于记录内部的 h2 > a 文本
name = card.select_one("h2 a").get_text(strip=True)

# status 位于记录内部的 span.badge 文本
status = card.select_one("span.badge").get_text(strip=True)

print("company_id:", company_id)
print("name:", name)
print("status:", status)

company_id: C000001
name: 临港精工001有限公司
status: 运营中


In [12]:
industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print("industry:", industry)
print("zone:", zone)
print("device_count:", device_count, type(device_count))

industry: 汽车零部件
zone: 智造一区
device_count: 6 <class 'int'>


In [13]:
# 完成第一页循环
companies = []

for card in cards:
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    companies.append({
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one("span.badge").get_text(strip=True),
    })

In [14]:
print("本页企业数：", len(companies))

本页企业数： 20


In [15]:
# 设计一个稳定的 selector，并验证：
next_link = soup.select_one("nav.pagination a")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [16]:
# 如果 href 是相对路径：
from urllib.parse import urljoin

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

https://115.159.159.82/companies?page=2


In [17]:
# 把下面三个步骤组合到循环中，连续采集 3 页
companies = []
current_url = base_url + "/companies"

for page_no in range(1, 4):

    # ① 请求当前页
    r = requests.get(current_url, timeout=10, verify=False)
    page_soup = BeautifulSoup(r.text, "html.parser")

    # ② 解析本页企业
    rows = []
    for card in page_soup.select("article.company-card"):
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = \
                item.select_one("dd").get_text(strip=True)

        rows.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace("台", "").strip()),
            "status": card.select_one("span.badge").get_text(strip=True),
        })

    companies.extend(rows)

    # ← 这里打印手册要求的格式
    print(f"第{page_no}页: {len(rows)}条")

    # ③ 找下一页
    next_link = next(
        (a for a in page_soup.select("nav.pagination a") if "下一页" in a.get_text()),
        None,
    )
    if next_link is None:
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数:", len(companies))

第1页: 20条


第2页: 20条


第3页: 20条
Company 总数: 60


In [18]:
print("唯一 company_id:", len(set(x["company_id"] for x in companies)))
print("Company 总数  :", len(companies))


唯一 company_id: 60
Company 总数  : 60


In [19]:
# 验证采集结果，数量与 ID 唯一性
print("companies:", len(companies))
print(
    "unique company_id:",
    len(set(x["company_id"] for x in companies))
)

print("devices:", len(devices))
print(
    "unique device_id:",
    len(set(x["device_id"] for x in devices))
)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


In [20]:
# Company–Device 一致性：用全部 Device 的 company_id 统计每家企业实际设备数
from collections import Counter

device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]          # Company 页面声称的设备数
    actual = device_counts.get(cid, 0)          # Device 实际统计到的设备数

    if expected != actual:
        mismatch_count += 1

print("不一致企业数:", mismatch_count)


不一致企业数: 0


In [21]:
# 保存采集结果
from pathlib import Path
import json

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录


In [22]:
# 最终验证结果
print("Lab1 最终验证结果：")
print("Company 总数                :", len(companies))
print("唯一 company_id             :", len(set(x["company_id"] for x in companies)))
print("Device 总数                 :", len(devices))
print("唯一 device_id              :", len(set(x["device_id"] for x in devices)))
print("Company-Device 不一致企业数  :", mismatch_count)

Lab1 最终验证结果：
Company 总数                : 60
唯一 company_id             : 60
Device 总数                 : 5000
唯一 device_id              : 5000
Company-Device 不一致企业数  : 0
